# Transformer (BERT-based) — Solution 2 to Problem 10.3

**Transformer architecture** (Vaswani et al., 2017) for electric-load forecasting, implemented in PyTorch with a `BertModel` encoder.

**Result reported in the book: Test MAPE = 0.01377333, Test RMSE = 62.7791.**

Outputs: actuals-vs-predictions (Fig. 10.5) and residual ACF (Fig. 10.6).

> **Note on figures.** This notebook contains the **exact code from the book**, preserved verbatim. The two plots are produced by the `plt.show()` calls at the end when the notebook is executed in an environment with PyTorch + the Transformers package installed (book versions: torch 2.7.1+cu128, transformers 4.53.0; a GPU is recommended). The figure images are not embedded here because this PyTorch model could not be re-executed during packaging.

## Requirements
```
pip install torch transformers scikit-learn PythonTsa
```
Book environment: torch 2.7.1+cu128, transformers 4.53.0.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, BertConfig
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (mean_squared_error,
                                mean_absolute_percentage_error)
import matplotlib.pyplot as plt
from PythonTsa.plot_acf_pacf import acf_pacf_fig
from PythonTsa.datadir import getdtapath

In [ ]:
# Load and preprocess raw data:
dtapath = getdtapath()
tsdta = pd.read_csv(dtapath + 'elec-temp.csv')
tsdta['time'] = pd.to_datetime(tsdta['time'])
tsdta.set_index('time', inplace=True)
valid_stdta = '2014-09-01 00:00:00'
test_stdta = '2014-11-01 00:00:00'
train_mask = tsdta.index < valid_stdta
valid_mask = (tsdta.index>=valid_stdta)&(tsdta.index<test_stdta)
test_mask = tsdta.index >= test_stdta
train_dat = tsdta[train_mask]
valid_dat = tsdta[valid_mask]
test_dat = tsdta[test_mask]
scaler = MinMaxScaler()
train_dat_scaled = scaler.fit_transform(train_dat)
valid_dat_scaled = scaler.transform(valid_dat)
test_dat_scaled = scaler.transform(test_dat)

In [ ]:
# Define Dataset(Torch tensor):
class TimeSeriesDataset(Dataset):
        def __init__(self, data, seq_length):
            self.data = data
            self.seq_length = seq_length
        def __len__(self):
            return len(self.data) - self.seq_length
        def __getitem__(self, idx):
            x = self.data[idx:idx + self.seq_length]
            y = self.data[idx + self.seq_length, 0]
            return torch.tensor(x, dtype=torch.float32),
                   torch.tensor(y, dtype=torch.float32)
# Let 2 torch.tensor()s in the same line when running

In [ ]:
# Define the Transformer model using BertModel:
class TransformerForecast(nn.Module):
        def __init__(
             self,
             input_dim,
             d_model,nhead,
             num_layers,
             dim_feedforward
        ):
             super(TransformerForecast, self).__init__()
             self.embedding = nn.Linear(input_dim, d_model)
             config = BertConfig(
                  hidden_size=d_model,
                  num_hidden_layers=num_layers,
                  num_attention_heads=nhead,
                  intermediate_size=dim_feedforward
             )
             self.bert = BertModel(config)
             self.fc = nn.Linear(d_model, 1)
        def forward(self, x):
            x = self.embedding(x)
            outputs = self.bert(inputs_embeds=x)
            x = outputs.last_hidden_state[:, -1, :]
            output = self.fc(x)
            return output.squeeze()

In [ ]:
# Define early stopping:
class EarlyStopping:
        def __init__(self, patience=3, min_delta=0):
            self.patience = patience
            self.min_delta = min_delta
            self.counter = 0
            self.best_loss = float('inf')
            self.early_stop = False
            self.best_model = None

        def __call__(self, val_loss, model=None):
            if self.best_loss - val_loss > self.min_delta:
                self.best_loss = val_loss
                self.counter = 0
                if model:
                    self.best_model = model.state_dict().copy()
            else:
                self.counter += 1
                if self.counter >= self.patience:
                    self.early_stop = True
early_stopping = EarlyStopping(patience=15, min_delta=0.0001)

In [ ]:
# Set hyperparameters:
seq_length = 24
batch_size = 48
d_model = 72
nhead = 4
num_layers = 6
dim_feedforward = 128
learning_rate = 1e-4
num_epochs = 70

In [ ]:
# Set Dataset and DataLoader:
train_dataset = TimeSeriesDataset(train_dat_scaled, seq_length)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size,
                                  shuffle=True)
valid_dataset = TimeSeriesDataset(valid_dat_scaled, seq_length)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size,
                                  shuffle=False)
test_dataset = TimeSeriesDataset(test_dat_scaled, seq_length)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size,
                                 shuffle=False)

In [ ]:
# Set model:
model = TransformerForecast(
         input_dim=2, d_model=d_model, nhead=nhead,
         num_layers=num_layers, dim_feedforward=dim_feedforward)

In [ ]:
# If CUDA is available, use it:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# Learning rate scheduler:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10)

In [ ]:
# Train model:
for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for inputs, targets in train_dataloader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        valid_loss = 0
        with torch.no_grad():
            for inputs, targets in valid_dataloader:
                inputs = inputs.to(device)
                targets = targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                valid_loss += loss.item()
        early_stopping(valid_loss/len(valid_dataloader), model)
        scheduler.step(valid_loss/len(valid_dataloader))
        if early_stopping.early_stop:
           print(f"Early stopping at epoch {epoch+1}")
           if early_stopping.best_model:
              model.load_state_dict(early_stopping.best_model)
           break
        print(f'Epoch {epoch + 1}/{num_epochs}, Train Loss:
              {train_loss / len(train_dataloader):.8f},'
              f'Valid Loss: {valid_loss / len(valid_dataloader):.8f}')
'''
make
f'Epoch {epoch + 1}/{num_epochs}, Train Loss:
{train_loss / len(train_dataloader):.8f},'
in the same line when running
'''
Epoch 1/70, Train Loss: 0.01168038,Valid Loss: 0.00276884
Epoch 2/70, Train Loss: 0.00421070,Valid Loss: 0.00336560
Epoch 3/70, Train Loss: 0.00291350,Valid Loss: 0.00108626
......
Epoch 33/70, Train Loss: 0.00052925,Valid Loss: 0.00028191
Epoch 34/70, Train Loss: 0.00051490,Valid Loss: 0.00026736
Epoch 35/70, Train Loss: 0.00049888,Valid Loss: 0.00029387
Early stopping at epoch 36

In [ ]:
# Predict:
model.eval()
predictions = []
true_values = []
with torch.no_grad():
        for inputs, targets in test_dataloader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            predictions.extend(outputs.cpu().numpy())
            true_values.extend(targets.cpu().numpy())

In [ ]:
# Inverse scaling:
predictions = np.array(predictions).reshape(-1, 1)
dummy_array = np.zeros((len(predictions), 2))
dummy_array[:, 0] = predictions.flatten()
original_scale_preds = scaler.inverse_transform(
                            dummy_array)[:, 0]
true_values = np.array(true_values).reshape(-1, 1)
dummy_array_true = np.zeros((len(true_values), 2))
dummy_array_true[:, 0] = true_values.flatten()
original_scale_actuals = scaler.inverse_transform(
                              dummy_array_true)[:, 0]

In [ ]:
# Calculate metrics:
mape = mean_absolute_percentage_error(original_scale_actuals,
                                           original_scale_preds)
mse = mean_squared_error(original_scale_actuals,
                              original_scale_preds)
rmse = np.sqrt(mse)
print(f"Test MAPE: {mape:.8f}, Test RMSE: {rmse:.4f}")
Test MAPE: 0.01377333, Test RMSE: 62.7791

In [ ]:
# Visualize:
test_index = test_dat.index[seq_length:]
original_scale_actuals = pd.Series(original_scale_actuals,
                                       index=test_index)
original_scale_preds = pd.Series(original_scale_preds,
                                      index=test_index)
plt.figure()
ax = original_scale_actuals[:144].plot(
          label='Actual load', style='-b')
original_scale_preds[:144].plot(ax = ax,
                        label='TR-Predicted load',style='--r')
plt.ylabel('Electricity load')
plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=3)
plt.savefig('transformer_actual_vs_pred.png', transparent=True, bbox_inches='tight')
plt.show()
resid = original_scale_actuals - original_scale_preds
acf_pacf_fig(resid, lag=72)
plt.show()